# Waveform Exploration

Configurable delay–Doppler and residual-CFO experiments: select one basis, inspect its
structure, then evaluate error rates.

**Load or train → inspect → evaluate.** One Learned per run; no per-SNR switching.
[Methods & controls](support/docs/CONFIGURABLE.md) · [Figure guide](support/docs/FIGURE_GUIDE.md)

## 1. Configure the experiment

Edit the grouped cell below, restart, **Run All**. Evaluation settings test a frozen W;
training settings create a new W. Saved replay accepts only its exact archived scenario.
Existing results need the [provided local bundles](support/docs/ARTIFACTS.md); new DD training does not.

In [ ]:
# CONFIGURATION — edit here; restart and Run All.
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "waveforms").is_dir() and (p / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT))
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[key] = "1"
os.environ["MPLCONFIGDIR"] = str(ROOT / "local/.matplotlib")

# 1. Experiment and basis
SCENARIO = "dd_v1"
MODE = "load_existing"  # load_existing / train_new (DD only).
EVIDENCE_MODE = "saved_replay"  # saved_replay / fresh; train_new and CFO require fresh.
N = 64  # DD: 64/128/256; CFO: 64.
K_FRACTION = 3 / 8
K = round(N * K_FRACTION)
if N * K_FRACTION != K:
    raise ValueError("N*K_FRACTION must be integral")
SUPPORT_POLICY = "frozen"
MODEL = dict(n=N, paths=3, delay_range=(0.0, 4.0), doppler_range=(-0.75, 0.75))

# 2. Local inputs and outputs
SCALING_ARTIFACT_DIR = ROOT / "local/scaling-artifacts"
SCALING_BUNDLE = ROOT / "local/advisor-scaling-artifacts.tar.gz"
ARTIFACT_DIR = ROOT / "local/artifacts"
ARTIFACT_BUNDLE = ROOT / "local/advisor-n64-artifacts.tar.gz"
CHECKPOINT = None  # local/*.npz: load, or new training save; None uses defaults.
OUTPUT_ROOT = ROOT / "local/notebook_runs"
SAVE_RUN = True

# 3. Figures — display only
BASIS_COLUMNS = {64:(0,11,15,42,44,52),128:(0,8,22,46),256:(0,40,87,148)}
EXAMPLE_COLUMNS = {64:(0,15,44),128:(0,8,22,46),256:(0,40,87,148)}
DISPLAY = dict(basis_columns=BASIS_COLUMNS.get(N,(0,)),
               example_columns=EXAMPLE_COLUMNS.get(N,(0,)), snr_max_db=25.0)
if MODE == "train_new" or CHECKPOINT is not None or SUPPORT_POLICY == "recalibrate":
    DISPLAY = dict(basis_columns=(0,1), example_columns=(0,), snr_max_db=25.0)
OVERLAY8_COLUMNS = tuple(range(0,N,max(1,N//8)))
HALF_COLUMNS = tuple(range(0,N,2))
PLOT_VARIANTS = ("overlay8_complex","overlay_half_complex",
                 "overlay8_energy_magnitude","overlay_half_energy_magnitude","grouped_spectra")
REVIEW = dict(variants=PLOT_VARIANTS,overlay8_columns=OVERLAY8_COLUMNS,half_columns=HALF_COLUMNS)

# 4. Fresh test evaluation — never retrains W
EVALUATION = None
if EVIDENCE_MODE == "fresh":
    EVALUATION = dict(seed=303, matrix_channels=2, matrix_index=0,
        max_channels=8, min_channels=4, batch_channels=2, blocks_per_channel=2,
        snrs_db=(0.0,10.0,18.0,25.0), modulations=("QPSK","16QAM"),
        target_errors=100, max_seconds_per_point=30.0,
        diagnostic_atoms=8, diagnostic_channels=8)

# 5. DD training — independent fit and validation banks
TRAINING = None
if MODE == "train_new":
    TRAINING = dict(model=dict(MODEL), validation_model=dict(MODEL),
        seed=101, validation_seed=202, iterations=20, learning_rate=0.02,
        layers=4, initialization="OFDM", batch_channels=8, validation_channels=8,
        validate_every=5, max_seconds=120.0, group_threshold=0.15)

# 6. Optional support recalibration for a loaded W
DISCOVERY = None
if SUPPORT_POLICY == "recalibrate":
    DISCOVERY = dict(model=dict(MODEL), seed=404, channels=16, group_threshold=0.15)

# 7. Residual CFO — distinct rectangular W/R model
if SCENARIO == "residual_cfo":
    MODEL = dict(n=N, streams=45, cfo_range=(-0.1,0.1))
    K, SUPPORT_POLICY, DISCOVERY, REVIEW = None, None, None, None
    DISPLAY = dict(basis_columns=(0,22,44), snr_max_db=25.0)
    if EVALUATION is not None:
        EVALUATION = {key:value for key,value in EVALUATION.items()
                      if key not in ("diagnostic_atoms","diagnostic_channels")}

## 2. Run overview

Settings are validated before work starts. New DD fits select one checkpoint on held-out
validation; the test bank cannot select W. **CFO train_new is unsupported.**
The table below identifies the active model, basis source and arithmetic.

In [ ]:
from waveforms.entry import start_workflow
from waveforms.presentation import (
    show_run_summary, show_channel_details, show_figure, show_coupling,
    show_examples, show_overlay_key, show_ber_summary, show_run_record,
)

workflow = None  # A failed selection cannot reuse an earlier run.
workflow = start_workflow(scenario=SCENARIO,mode=MODE,evidence_mode=EVIDENCE_MODE,
    model=MODEL,display=DISPLAY,evaluation=EVALUATION,training=TRAINING,
    k=K,support_policy=SUPPORT_POLICY,discovery=DISCOVERY,checkpoint=CHECKPOINT,
    workspace=ROOT,output_root=OUTPUT_ROOT,save_run=SAVE_RUN,
    scaling_artifacts=SCALING_ARTIFACT_DIR,scaling_bundle=SCALING_BUNDLE,
    artifacts=ARTIFACT_DIR,bundle=ARTIFACT_BUNDLE,review=REVIEW)
show_run_summary(workflow)

## 3. Physical and effective channel

DD uses normalized multipath H and **A = WᴴHW**. Residual CFO uses positive phase
rotation and **A = RHW**, with W[64,45] and distinct R[45,64]; its analysis noise covariance
is σ²RRᴴ. Heatmaps show entry power; complex matrices remain available in memory/files.

In [ ]:
channels, A = workflow.generate_channels()
H, Hraw, W = channels.h, channels.raw, workflow.ws
show_channel_details(workflow)
workflow.show("channel_matrices_native")

## 4. Waveform structure

Time and frequency plots retain complex real/imaginary coefficients. DD views link
coupling groups back to native basis IDs; CFO shows its distinct transmit/receive columns.
Dense spectra interpolate the finite block—they add no physical resolution.

In [ ]:
show_figure(workflow, "basis_time", "4.1 Time-domain bases",
            "Same native IDs across Learned, OTFS and AFDM; CFO compares W and Rᴴ columns.")

In [ ]:
show_figure(workflow, "basis_frequency", "4.2 Complex frequency-domain bases",
            "Real/imaginary DTFT coefficients with 1/√N normalization; no peak scaling.")

In [ ]:
show_coupling(workflow)

In [ ]:
show_figure(workflow, "grouped_spectra", "4.4 Group member spectra",
            "Every member’s absolute DFT energy and black group mean, beside individually peak-normalized magnitude (dB).")

In [ ]:
show_examples(workflow)

In [ ]:
show_figure(workflow, "coupling_propagation", "4.6 Received basis propagation",
            "One transmit column through the same H. Complex group contributions reconstruct the signal; their powers are not additive.")

## 5. Overlay comparisons

Enabled DD overlays retain all **8** or **N/2** selected bases together, with complete
native-ID/group/rank keys. Dense curves can overlap; individual examples above remain
traceable. These display choices change neither W nor BER. CFO has no DD overlay model.

In [ ]:
show_overlay_key(workflow)

In [ ]:
show_figure(workflow, "overlay8_complex", "5.1 Eight bases · complex",
            "Real/imaginary time samples and complex frequency coefficients.")

In [ ]:
show_figure(workflow, "overlay_half_complex", "5.2 N/2 bases · complex",
            "Same complex representation with every selected curve retained.")

In [ ]:
show_figure(workflow, "overlay8_energy_magnitude", "5.3 Eight bases · energy and magnitude",
            "Absolute DFT energy versus individually peak-normalized spectral magnitude.")

In [ ]:
show_figure(workflow, "overlay_half_energy_magnitude", "5.4 N/2 bases · energy and magnitude",
            "Same energy/magnitude definitions; normalization compares shape, not absolute strength.")

## 6. Error-rate evaluation

Replay loads saved counts; **fresh** simulates independent test observations. DD shares
H, bits, physical noise, K and stopping prefixes across methods; fixed/full receivers stay
separate. CFO uses direct R analysis. Intervals are approximate under stopping; zero errors
remain unresolved. The new trainer is a surrogate fit, with limited trials—not a BER-win claim.

In [ ]:
ber = show_ber_summary(workflow)

In [ ]:
show_figure(workflow, "ber_QPSK_raw", "6.1 QPSK",
            "Uncoded raw slicing, Es/N0. Zero-error markers show resolution at 1/bits; bounds and counts are in the table above.")

In [ ]:
show_figure(workflow, "ber_16QAM_raw", "6.2 16QAM",
            "Normalized Gray 16QAM under the same receiver and evidence rules.")

## 7. Run record

Local archives preserve full settings, selection, complex arrays, counts, mappings and
source hashes. New training saves an inference checkpoint when archiving is enabled.
This notebook demonstrates the workflow; it does not establish production precision,
training-seed uncertainty or physical-link performance.

In [ ]:
result = workflow.finish()
show_run_record(workflow, result)